In [ ]:
# Bond Portfolio Tracker

Sequential event-driven portfolio engine.

| Component | Purpose |
|---|---|
| **`EventReader`** | Loads all trade events; exposes them one at a time, tracking the current stream position |
| **`PortfolioManager`** | Maintains per-bond positions and P&L; updates incrementally on each event; supports multiple query views |


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_columns", 20)

DATA_DIR = Path("data")


In [2]:
# ── Reference data: bonds ─────────────────────────────────────────────────────
# Row 0 of the CSV is an empty separator; header is on row 1
bonds_df = pd.read_csv(DATA_DIR / "bonds - bonds.csv", header=1).set_index("BondID")

# Accrued interest per 100 face: AI = annual_coupon_rate × (months_since_coupon / 12) × 100
bonds_df["AccruedInterest"] = bonds_df["Coupon"] * bonds_df["MonthsSinceCoupon"] / 12 * 100

print("Bond reference data")
display(bonds_df)


Bond reference data


,Coupon,Frequency,MonthsSinceCoupon,AccruedInterest
BondID,,,,
BOND1,0.0500,2,3,1.2500
BOND2,0.0400,1,6,2.0000
BOND3,0.0600,2,4,2.0000
BOND4,0.0300,1,6,1.5000
BOND5,0.0700,2,1,0.5833


In [3]:
class EventReader:
    """
    Sequential reader over trade events.

    The reader loads all events once at initialisation and exposes a
    pointer-based interface so callers can consume them one at a time
    while always knowing exactly where they are in the stream.

    Core interface
    --------------
    reader.current_event_id   EventID of the next unconsumed event
    reader.events_processed   How many events have been consumed
    reader.events_remaining   How many are left
    reader.peek()             Inspect next event without consuming it
    reader.next_event()       Consume and return the next event
    reader.reset()            Rewind stream to the beginning
    """

    def __init__(self, filepath: str | Path):
        self._events  = pd.read_csv(filepath).set_index("EventID")
        self._ids     = self._events.index.tolist()
        self._pointer = 0                        # index of the next event to yield

    # ── State ─────────────────────────────────────────────────────────────────

    @property
    def current_event_id(self):
        """EventID that will be returned on the next call to next_event()."""
        return self._ids[self._pointer] if not self.is_exhausted else None

    @property
    def events_processed(self) -> int:
        return self._pointer

    @property
    def events_remaining(self) -> int:
        return len(self._ids) - self._pointer

    @property
    def is_exhausted(self) -> bool:
        return self._pointer >= len(self._ids)

    # ── Stream operations ─────────────────────────────────────────────────────

    def peek(self) -> pd.Series | None:
        """Return the next event as a Series without advancing the pointer."""
        if self.is_exhausted:
            return None
        eid = self._ids[self._pointer]
        row = self._events.loc[eid].copy()
        row.name = eid
        return row

    def next_event(self) -> pd.Series | None:
        """Consume and return the next event, advancing the pointer by one."""
        if self.is_exhausted:
            return None
        eid = self._ids[self._pointer]
        row = self._events.loc[eid].copy()
        row.name = eid
        self._pointer += 1
        return row

    def remaining_events(self) -> pd.DataFrame:
        """All unconsumed events as a DataFrame (does not advance the pointer)."""
        return self._events.iloc[self._pointer :].copy()

    def reset(self) -> "EventReader":
        """Rewind to the beginning of the stream."""
        self._pointer = 0
        return self

    # ── Python protocol ───────────────────────────────────────────────────────

    def __iter__(self):
        return self

    def __next__(self):
        event = self.next_event()
        if event is None:
            raise StopIteration
        return event

    def __len__(self):
        return len(self._ids)

    def __repr__(self):
        return (
            f"EventReader(total={len(self._ids)}, "
            f"processed={self._pointer}, "
            f"remaining={self.events_remaining}, "
            f"next_id={self.current_event_id})"
        )


In [4]:
class PortfolioManager:
    """
    Maintains bond positions and P&L incrementally as trade events arrive.

    Positions track net quantity (positive = long, negative = short) and a
    weighted-average clean cost basis.  Realized P&L is booked whenever a
    position is reduced or reversed; unrealized P&L is marked to the latest
    trade price.

    Full price  = clean price + accrued interest (static from bonds reference).
    Present value (PV) = net_qty × full_price / 100.

    All keyword arguments are validated; ValueError is raised for unknown
    bond IDs, desks, traders, or event IDs.

    Preprocessing / O(1) snapshots
    --------------------------------
    Every call to process_event() automatically captures an immutable snapshot
    of the full portfolio state at that event.  Alternatively, call
    preprocess_all(reader) to bulk-process all remaining events at once.

    After at least one event has been processed, every query method accepts an
    optional  at_event_id  keyword.  Passing it returns the portfolio state
    *after* that specific event in O(1) time — no event replay required.

    Query methods (all accept at_event_id=None for current state)
    -------------
    get_positions(bond_id, at_event_id)         Active (non-zero) positions
    get_all_positions(at_event_id)              All positions, including flat bonds
    get_summary(at_event_id)                    Portfolio-level aggregate metrics
    get_trade_history(bond_id, desk, trader)    Full trades ledger (always current)
    get_by_desk(desk, at_event_id)              Notional and P&L grouped by desk
    get_by_trader(desk, trader, at_event_id)    Notional and P&L grouped by desk + trader
    get_bond_snapshot(bond_id, at_event_id)     Position, price, and PV for a single bond
    get_pv(bond_id, trader, at_event_id)        PV scoped by bond / trader / all positions
    get_pv_by_desk(at_event_id)                 Total PV of net positions grouped by desk
    get_pnl_since(from_event_id, at_event_id)   Realized P&L in a window + current unrealized
    """

    FACE = 100  # bonds are quoted per 100 face value

    def __init__(self, bonds_df: pd.DataFrame):
        self._bonds   = bonds_df.copy()
        bond_ids      = bonds_df.index.tolist()

        # ── Live positions table ──────────────────────────────────────────────
        cols = [
            "NetQty", "AvgCostClean", "LastCleanPrice",
            "AccruedInterest", "FullPrice",
            "MarketValue", "RealizedPnL", "UnrealizedPnL", "TotalPnL",
        ]
        self._positions = pd.DataFrame(
            np.nan, index=pd.Index(bond_ids, name="BondID"), columns=cols
        )
        self._positions[["NetQty", "RealizedPnL"]] = 0.0
        self._trades: list[dict] = []

        # ── Running aggregation state (O(1) per-event updates) ────────────────
        # These mirror what used to be recomputed on every group-by query.
        self._run_cumreal: dict[str, float]         = {b: 0.0 for b in bond_ids}
        self._run_desk:    dict[str, dict]          = {}   # desk -> counters
        self._run_trader:  dict[tuple, dict]        = {}   # (desk, trader) -> counters
        self._run_tqty:    dict[str, dict[str, float]] = {}  # trader -> bond -> net qty
        self._run_dqty:    dict[str, dict[str, float]] = {}  # desk   -> bond -> net qty

        # ── Snapshot store: event_id -> state after that event ────────────────
        self._pos_snaps:     dict[int, pd.DataFrame]             = {}
        self._cumreal_snaps: dict[int, dict[str, float]]         = {}
        self._desk_snaps:    dict[int, pd.DataFrame]             = {}
        self._trader_snaps:  dict[int, pd.DataFrame]             = {}
        self._tqty_snaps:    dict[int, dict[str, dict[str, float]]] = {}
        self._dqty_snaps:    dict[int, dict[str, dict[str, float]]] = {}
        self._event_order:   list[int]                           = []
        self._event_idx:     dict[int, int]                      = {}  # event_id -> index in _event_order

    # ── Snapshot helpers ──────────────────────────────────────────────────────

    @staticmethod
    def _agg_to_df(agg: dict, index_name) -> pd.DataFrame:
        cols = ["NumTrades", "GrossNotional", "NetNotional", "RealizedPnL"]
        if not agg:
            return pd.DataFrame(columns=cols)
        df = pd.DataFrame.from_dict(agg, orient="index")[cols]
        if isinstance(index_name, list):
            df.index = pd.MultiIndex.from_tuples(list(df.index), names=index_name)
        else:
            df.index.name = index_name
        return df

    def _capture_snapshot(self, event_id: int) -> None:
        """Store an immutable copy of all portfolio state after event_id."""
        self._pos_snaps[event_id]     = self._positions.copy()
        self._cumreal_snaps[event_id] = self._run_cumreal.copy()
        self._desk_snaps[event_id]    = self._agg_to_df(self._run_desk,   "Desk")
        self._trader_snaps[event_id]  = self._agg_to_df(self._run_trader, ["Desk", "Trader"])
        self._tqty_snaps[event_id]    = {t: d.copy() for t, d in self._run_tqty.items()}
        self._dqty_snaps[event_id]    = {d: b.copy() for d, b in self._run_dqty.items()}
        self._event_idx[event_id]     = len(self._event_order)
        self._event_order.append(event_id)

    def _cumreal_before(self, event_id: int) -> dict[str, float]:
        """Cumulative realized PnL per bond BEFORE event_id was processed. O(1)."""
        idx = self._event_idx[event_id]
        if idx == 0:
            return {b: 0.0 for b in self._bonds.index}
        return self._cumreal_snaps[self._event_order[idx - 1]]

    def _resolve_snap(self, at_event_id: int | None) -> int | None:
        """Validate and return the snapshot key; None means use live state."""
        if at_event_id is None:
            return None
        if not self._pos_snaps:
            raise RuntimeError(
                "No snapshots available. Process at least one event first."
            )
        if at_event_id not in self._pos_snaps:
            raise ValueError(
                f"No snapshot for EventID {at_event_id}. "
                f"Available: {sorted(self._pos_snaps)}"
            )
        return at_event_id

    # ── Validation helpers ────────────────────────────────────────────────────

    def _check_bond(self, bond_id: str) -> None:
        valid = self._bonds.index.tolist()
        if bond_id not in valid:
            raise ValueError(f"Unknown bond '{bond_id}'. Valid bond IDs: {valid}")

    def _check_desk(self, desk: str) -> None:
        valid = sorted(self._run_desk)
        if desk not in valid:
            raise ValueError(f"Unknown desk '{desk}'. Valid desks: {valid}")

    def _check_trader(self, trader: str) -> None:
        valid = sorted(self._run_tqty)
        if trader not in valid:
            raise ValueError(f"Unknown trader '{trader}'. Valid traders: {valid}")

    def _check_event_id(self, event_id: int) -> None:
        if event_id not in self._event_idx:
            raise ValueError(
                f"EventID {event_id} not in processed events: "
                f"{sorted(self._event_idx)}"
            )

    # ── Core update ───────────────────────────────────────────────────────────

    def process_event(self, event: pd.Series) -> "PortfolioManager":
        """
        Process one trade event, update live positions, and capture a snapshot.
        Returns self so calls can be chained.
        """
        bond_id    = event["BondID"]
        buy_sell   = event["BuySell"]
        qty        = float(event["Quantity"])
        clean_px   = float(event["CleanPrice"])
        desk       = event["Desk"]
        trader     = event["Trader"]
        event_id   = int(event.name)

        ai         = float(self._bonds.loc[bond_id, "AccruedInterest"])
        full_px    = clean_px + ai
        signed_qty = qty if buy_sell == "BUY" else -qty

        old_qty  = float(self._positions.loc[bond_id, "NetQty"])
        raw_avg  = self._positions.loc[bond_id, "AvgCostClean"]
        old_avg  = 0.0 if pd.isna(raw_avg) else float(raw_avg)
        new_qty  = old_qty + signed_qty
        real_pnl = 0.0
        new_avg  = old_avg

        if signed_qty > 0:                          # ── BUY ─────────────────
            if old_qty >= 0:
                new_avg = (old_qty * old_avg + qty * clean_px) / new_qty if new_qty else clean_px
            else:
                cover    = min(qty, abs(old_qty))
                real_pnl = (old_avg - clean_px) * cover
                if new_qty > 0:
                    new_avg = clean_px
                elif new_qty == 0:
                    new_avg = 0.0

        else:                                       # ── SELL ────────────────
            if old_qty <= 0:
                total_short = abs(old_qty) + qty
                new_avg = (abs(old_qty) * old_avg + qty * clean_px) / total_short if total_short else clean_px
            else:
                close    = min(qty, old_qty)
                real_pnl = (clean_px - old_avg) * close
                if new_qty < 0:
                    new_avg = clean_px
                elif new_qty == 0:
                    new_avg = 0.0

        unrealized     = (clean_px - new_avg) * new_qty if new_qty != 0 else 0.0
        prev_realized  = float(self._positions.loc[bond_id, "RealizedPnL"])
        total_realized = prev_realized + real_pnl
        notional       = signed_qty * full_px / self.FACE

        self._positions.loc[bond_id, "NetQty"]          = new_qty
        self._positions.loc[bond_id, "AvgCostClean"]    = new_avg if new_qty != 0 else np.nan
        self._positions.loc[bond_id, "LastCleanPrice"]  = clean_px
        self._positions.loc[bond_id, "AccruedInterest"] = ai
        self._positions.loc[bond_id, "FullPrice"]        = full_px
        self._positions.loc[bond_id, "MarketValue"]      = new_qty * full_px / self.FACE
        self._positions.loc[bond_id, "RealizedPnL"]      = total_realized
        self._positions.loc[bond_id, "UnrealizedPnL"]    = unrealized
        self._positions.loc[bond_id, "TotalPnL"]         = total_realized + unrealized

        self._trades.append({
            "EventID":         event_id,
            "Desk":            desk,
            "Trader":          trader,
            "BondID":          bond_id,
            "BuySell":         buy_sell,
            "Quantity":        qty,
            "SignedQty":       signed_qty,
            "CleanPrice":      clean_px,
            "AccruedInterest": ai,
            "FullPrice":       full_px,
            "Notional":        notional,
            "RealizedPnL":     real_pnl,
        })

        # ── Update running aggregations (O(1) per event) ──────────────────────
        self._run_cumreal[bond_id] += real_pnl

        for agg_dict, key in [
            (self._run_desk,   desk),
            (self._run_trader, (desk, trader)),
        ]:
            if key not in agg_dict:
                agg_dict[key] = {
                    "NumTrades": 0, "GrossNotional": 0.0,
                    "NetNotional": 0.0, "RealizedPnL": 0.0,
                }
            agg_dict[key]["NumTrades"]     += 1
            agg_dict[key]["GrossNotional"] += abs(notional)
            agg_dict[key]["NetNotional"]   += notional
            agg_dict[key]["RealizedPnL"]   += real_pnl

        if trader not in self._run_tqty:
            self._run_tqty[trader] = {b: 0.0 for b in self._bonds.index}
        self._run_tqty[trader][bond_id] += signed_qty

        if desk not in self._run_dqty:
            self._run_dqty[desk] = {b: 0.0 for b in self._bonds.index}
        self._run_dqty[desk][bond_id] += signed_qty

        # ── Capture immutable snapshot for this event ─────────────────────────
        self._capture_snapshot(event_id)
        return self

    # ── Preprocessing ─────────────────────────────────────────────────────────

    def preprocess_all(self, event_reader: "EventReader") -> "PortfolioManager":
        """
        Consume all remaining events from reader in one call, building a
        per-event snapshot for every event.  After this, all query methods
        support O(1) at_event_id lookups.  Returns self.
        """
        for event in event_reader:
            self.process_event(event)
        return self

    # ── Dispatch helper ───────────────────────────────────────────────────────

    def _pos_at(self, snap: int | None) -> pd.DataFrame:
        return self._pos_snaps[snap] if snap is not None else self._positions

    # ── Core query methods ────────────────────────────────────────────────────

    def get_positions(
        self,
        bond_id: str | None = None,
        at_event_id: int | None = None,
    ) -> pd.DataFrame:
        """Active (non-zero NetQty) positions. Optionally filter to one bond."""
        if bond_id is not None:
            self._check_bond(bond_id)
        snap = self._resolve_snap(at_event_id)
        df   = self._pos_at(snap)
        df   = df[df["NetQty"] != 0].copy()
        if bond_id:
            df = df.loc[[bond_id]] if bond_id in df.index else df.iloc[:0]
        return df

    def get_all_positions(self, at_event_id: int | None = None) -> pd.DataFrame:
        """All positions, including flat bonds."""
        snap = self._resolve_snap(at_event_id)
        return self._pos_at(snap).copy()

    def get_summary(self, at_event_id: int | None = None) -> pd.Series:
        """Portfolio-level aggregate metrics."""
        snap     = self._resolve_snap(at_event_id)
        p        = self._pos_at(snap)
        n_events = (self._event_idx[snap] + 1) if snap is not None else len(self._trades)
        return pd.Series(
            {
                "TotalMarketValue":   p["MarketValue"].sum(),
                "TotalRealizedPnL":   p["RealizedPnL"].sum(),
                "TotalUnrealizedPnL": p["UnrealizedPnL"].sum(),
                "TotalPnL":           p["TotalPnL"].sum(),
                "ActivePositions":    int((p["NetQty"] != 0).sum()),
                "LongPositions":      int((p["NetQty"] > 0).sum()),
                "ShortPositions":     int((p["NetQty"] < 0).sum()),
                "EventsProcessed":    n_events,
            },
            name="PortfolioSummary",
        )

    def get_trade_history(
        self,
        bond_id: str | None = None,
        desk:    str | None = None,
        trader:  str | None = None,
    ) -> pd.DataFrame:
        """Full trades ledger (always reflects current state; filter by bond/desk/trader)."""
        if bond_id is not None: self._check_bond(bond_id)
        if desk is not None:    self._check_desk(desk)
        if trader is not None:  self._check_trader(trader)
        if not self._trades:
            return pd.DataFrame()
        df = pd.DataFrame(self._trades).set_index("EventID")
        if bond_id: df = df[df["BondID"] == bond_id]
        if desk:    df = df[df["Desk"]   == desk]
        if trader:  df = df[df["Trader"] == trader]
        return df

    def get_by_desk(
        self,
        desk: str | None = None,
        at_event_id: int | None = None,
    ) -> pd.DataFrame:
        """Gross/net notional and realized P&L grouped by desk."""
        if not self._trades:
            return pd.DataFrame()
        if desk is not None:
            self._check_desk(desk)
        snap = self._resolve_snap(at_event_id)
        df   = (
            self._desk_snaps[snap].copy()
            if snap is not None
            else self._agg_to_df(self._run_desk, "Desk")
        )
        if desk is not None:
            df = df.loc[[desk]] if desk in df.index else df.iloc[:0]
        return df.sort_values("GrossNotional", ascending=False)

    def get_by_trader(
        self,
        desk:   str | None = None,
        trader: str | None = None,
        at_event_id: int | None = None,
    ) -> pd.DataFrame:
        """Gross/net notional and realized P&L grouped by desk + trader."""
        if not self._trades:
            return pd.DataFrame()
        if desk is not None:   self._check_desk(desk)
        if trader is not None: self._check_trader(trader)
        snap = self._resolve_snap(at_event_id)
        df   = (
            self._trader_snaps[snap].copy()
            if snap is not None
            else self._agg_to_df(self._run_trader, ["Desk", "Trader"])
        )
        if desk is not None:
            df = df.loc[df.index.get_level_values("Desk")   == desk]
        if trader is not None:
            df = df.loc[df.index.get_level_values("Trader") == trader]
        return df.sort_values("RealizedPnL", ascending=False)

    # ── Extended query methods ────────────────────────────────────────────────

    def get_bond_snapshot(
        self,
        bond_id: str,
        at_event_id: int | None = None,
    ) -> pd.Series:
        """Current (or historical) position, price, and PV for a single bond."""
        self._check_bond(bond_id)
        snap = self._resolve_snap(at_event_id)
        s = self._pos_at(snap).loc[bond_id, [
            "NetQty", "AvgCostClean", "LastCleanPrice",
            "AccruedInterest", "FullPrice", "MarketValue",
            "RealizedPnL", "UnrealizedPnL", "TotalPnL",
        ]].copy()
        s.name = bond_id
        return s

    def get_pv(
        self,
        bond_id: str | None = None,
        trader:  str | None = None,
        at_event_id: int | None = None,
    ) -> pd.DataFrame:
        """
        Present value view.  Pass at most one scope keyword:
          bond_id  → PV row for that bond
          trader   → per-bond PV for that trader's net signed quantity
          (neither)→ PV for all bonds with a non-zero net position
        """
        if bond_id is not None and trader is not None:
            raise ValueError("Provide at most one of bond_id or trader, not both.")
        snap = self._resolve_snap(at_event_id)
        pos  = self._pos_at(snap)

        if bond_id is not None:
            self._check_bond(bond_id)
            return (
                pos.loc[[bond_id], ["NetQty", "FullPrice", "MarketValue"]]
                .rename(columns={"MarketValue": "PV"})
                .copy()
            )

        if trader is not None:
            self._check_trader(trader)
            tqty = (
                self._tqty_snaps[snap]
                if snap is not None
                else self._run_tqty
            ).get(trader, {})
            rows = [
                {
                    "BondID":   b,
                    "NetQty":   q,
                    "FullPrice": pos.loc[b, "FullPrice"],
                    "PV":       q * pos.loc[b, "FullPrice"] / self.FACE,
                }
                for b, q in tqty.items()
                if q != 0
            ]
            return (
                pd.DataFrame(rows).set_index("BondID")
                if rows
                else pd.DataFrame(columns=["NetQty", "FullPrice", "PV"])
            )

        return (
            pos.loc[pos["NetQty"] != 0, ["NetQty", "FullPrice", "MarketValue"]]
            .rename(columns={"MarketValue": "PV"})
            .copy()
        )

    def get_pv_by_desk(self, at_event_id: int | None = None) -> pd.DataFrame:
        """Total PV of net positions attributed to each desk."""
        snap = self._resolve_snap(at_event_id)
        if snap is not None:
            dqty_store = self._dqty_snaps[snap]
            pos        = self._pos_snaps[snap]
        else:
            if not self._trades:
                return pd.DataFrame(columns=["TotalPV"])
            dqty_store = self._run_dqty
            pos        = self._positions
        rows = [
            {
                "Desk":    d,
                "TotalPV": sum(
                    q * pos.loc[b, "FullPrice"] / self.FACE
                    for b, q in bond_qtys.items()
                    if q != 0 and not pd.isna(pos.loc[b, "FullPrice"])
                ),
            }
            for d, bond_qtys in dqty_store.items()
        ]
        return (
            pd.DataFrame(rows).set_index("Desk").sort_values("TotalPV", ascending=False)
            if rows
            else pd.DataFrame(columns=["TotalPV"])
        )

    def get_pnl_since(
        self,
        from_event_id: int,
        at_event_id:   int | None = None,
    ) -> pd.DataFrame:
        """
        P&L breakdown from from_event_id (inclusive) through at_event_id
        (defaults to the latest processed event).
          RealizedPnL_Since – realized gains/losses booked in that window  [O(1)]
          UnrealizedPnL     – mark-to-market unrealized P&L at at_event_id [O(1)]
          TotalPnL          – sum of the two
        Only bonds with any non-zero P&L are shown.
        """
        if not self._trades:
            raise ValueError("No events have been processed yet.")
        self._check_event_id(from_event_id)

        # Resolve the "end" snapshot
        snap = self._resolve_snap(at_event_id) if at_event_id is not None else self._event_order[-1]
        if self._event_idx[from_event_id] > self._event_idx[snap]:
            raise ValueError(
                f"from_event_id {from_event_id} comes after at_event_id {snap}."
            )

        cum_at     = self._cumreal_snaps[snap]
        cum_before = self._cumreal_before(from_event_id)

        realized_since = pd.Series(
            {b: cum_at.get(b, 0.0) - cum_before.get(b, 0.0) for b in self._bonds.index},
            name="RealizedPnL_Since",
        )
        unreal = self._pos_snaps[snap]["UnrealizedPnL"].rename("UnrealizedPnL")
        result = pd.concat([realized_since, unreal], axis=1).fillna(0)
        result["TotalPnL"] = result["RealizedPnL_Since"] + result["UnrealizedPnL"]
        return result[result[["RealizedPnL_Since", "UnrealizedPnL", "TotalPnL"]].any(axis=1)]

    def __repr__(self):
        n = int((self._positions["NetQty"] != 0).sum())
        return (
            f"PortfolioManager("
            f"bonds={len(self._positions)}, "
            f"active_positions={n}, "
            f"events_processed={len(self._trades)}, "
            f"snapshots={len(self._pos_snaps)})"
        )


In [5]:
# ── Preprocess all events ─────────────────────────────────────────────────────
reader    = EventReader(DATA_DIR / "events - events.csv")
portfolio = PortfolioManager(bonds_df).preprocess_all(reader)
print(portfolio)


PortfolioManager(bonds=5, active_positions=5, events_processed=150, snapshots=150)


In [6]:
# ── Keyword query interface ───────────────────────────────────────────────────

_HELP = """\
Commands (append @N to query at historical event N):
  bond BOND1 [@N]        Position, price, and PV for a bond
  desk [NY] [@N]         Aggregated results for one / all desks
  trader [T_HK_2] [@N]  Aggregated results for one / all traders
  pnl 5 [12]            P&L from event 5 (optionally through event 12)
  pv [@N]               PV for all active positions
  pv BOND1 [@N]         PV for a specific bond
  pv T_HK_2 [@N]        PV for a specific trader
  pvdesk [@N]           Total PV grouped by desk
  summary [@N]          Portfolio-level summary
  positions [BOND1] [@N] Active positions (optionally one bond)
  help                  Show this message"""


def query(cmd: str) -> None:
    tokens = cmd.strip().split()
    if not tokens:
        raise ValueError("Empty query. Type query('help') for commands.")

    # Extract optional @event_id from anywhere in the token list
    at_event = None
    rest = []
    for t in tokens:
        if t.startswith("@"):
            try:
                at_event = int(t[1:])
            except ValueError:
                raise ValueError(f"Bad event reference '{t}'. Use @N where N is an integer.")
        else:
            rest.append(t)

    if not rest:
        raise ValueError("No command keyword found. Type query('help') for commands.")

    verb = rest[0].lower()
    args = rest[1:]

    if verb == "help":
        print(_HELP)

    elif verb == "bond":
        if len(args) != 1:
            raise ValueError("Usage: bond <BOND_ID> [@event]")
        bid = args[0].upper()
        display(portfolio.get_bond_snapshot(bid, at_event_id=at_event).to_frame(name=bid))

    elif verb == "desk":
        d = args[0].upper() if args else None
        display(portfolio.get_by_desk(desk=d, at_event_id=at_event))

    elif verb == "trader":
        t = args[0] if args else None
        display(portfolio.get_by_trader(trader=t, at_event_id=at_event))

    elif verb == "pnl":
        if not args:
            raise ValueError("Usage: pnl <FROM_EVENT> [TO_EVENT]")
        try:
            from_eid = int(args[0])
        except ValueError:
            raise ValueError(f"FROM_EVENT must be an integer, got '{args[0]}'.")
        to_eid = at_event
        if len(args) > 1:
            try:
                to_eid = int(args[1])
            except ValueError:
                raise ValueError(f"TO_EVENT must be an integer, got '{args[1]}'.")
        display(portfolio.get_pnl_since(from_event_id=from_eid, at_event_id=to_eid))

    elif verb == "pv":
        if not args:
            display(portfolio.get_pv(at_event_id=at_event))
        else:
            key = args[0]
            if key.upper().startswith("BOND"):
                display(portfolio.get_pv(bond_id=key.upper(), at_event_id=at_event))
            elif key.startswith("T_"):
                display(portfolio.get_pv(trader=key, at_event_id=at_event))
            else:
                raise ValueError(
                    f"Unknown PV target '{key}'. "
                    f"Use a bond ID (BOND…) or trader ID (T_…)."
                )

    elif verb == "pvdesk":
        display(portfolio.get_pv_by_desk(at_event_id=at_event))

    elif verb == "summary":
        display(portfolio.get_summary(at_event_id=at_event).to_frame(name="Value"))

    elif verb == "positions":
        bid = args[0].upper() if args else None
        display(portfolio.get_positions(bond_id=bid, at_event_id=at_event))

    else:
        raise ValueError(f"Unknown command '{verb}'. Type query('help') for commands.")


In [ ]:
# ── Interactive Query UI ──────────────────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display, clear_output

_bonds   = sorted(portfolio._bonds.index)
_desks   = sorted(portfolio._run_desk)
_traders = sorted(portfolio._run_tqty)
_events  = sorted(portfolio._event_idx)

# ── Widgets ───────────────────────────────────────────────────────────────────
cmd_dd = widgets.Dropdown(
    options=["bond", "desk", "trader", "pnl", "pv", "pvdesk", "summary", "positions"],
    value="summary",
    description="Command:",
)
arg1_dd = widgets.Dropdown(options=["(none)"], value="(none)", description="Arg 1:")
arg2_dd = widgets.Dropdown(options=["(none)"], value="(none)", description="Arg 2:")
event_dd = widgets.Dropdown(
    options=["latest"] + [str(e) for e in _events],
    value="latest",
    description="At event:",
)
run_btn = widgets.Button(description="Run query", button_style="primary", icon="play")
output  = widgets.Output()

# ── Dynamic option updates ────────────────────────────────────────────────────
def _update_options(*_):
    cmd = cmd_dd.value
    if cmd == "bond":
        arg1_dd.options = _bonds
        arg1_dd.value   = _bonds[0]
        arg1_dd.layout.display = ""
        arg2_dd.layout.display = "none"
    elif cmd == "desk":
        arg1_dd.options = ["(all)"] + _desks
        arg1_dd.value   = "(all)"
        arg1_dd.layout.display = ""
        arg2_dd.layout.display = "none"
    elif cmd == "trader":
        arg1_dd.options = ["(all)"] + _traders
        arg1_dd.value   = "(all)"
        arg1_dd.layout.display = ""
        arg2_dd.layout.display = "none"
    elif cmd == "pnl":
        arg1_dd.options = [str(e) for e in _events]
        arg1_dd.value   = str(_events[0])
        arg1_dd.description = "From:"
        arg2_dd.options = ["(latest)"] + [str(e) for e in _events]
        arg2_dd.value   = "(latest)"
        arg2_dd.description = "To:"
        arg1_dd.layout.display = ""
        arg2_dd.layout.display = ""
    elif cmd == "pv":
        arg1_dd.options = ["(all)"] + _bonds + _traders
        arg1_dd.value   = "(all)"
        arg1_dd.layout.display = ""
        arg2_dd.layout.display = "none"
    else:  # pvdesk, summary, positions
        arg1_dd.layout.display = "none"
        arg2_dd.layout.display = "none"
    if cmd != "pnl":
        arg1_dd.description = "Arg 1:"
        arg2_dd.description = "Arg 2:"

cmd_dd.observe(_update_options, names="value")
_update_options()

# ── Run handler ───────────────────────────────────────────────────────────────
def _run(_):
    output.clear_output(wait=True)
    with output:
        cmd = cmd_dd.value
        at  = None if event_dd.value == "latest" else int(event_dd.value)
        try:
            if cmd == "bond":
                query(f"bond {arg1_dd.value}" + (f" @{at}" if at else ""))
            elif cmd == "desk":
                d = "" if arg1_dd.value == "(all)" else f" {arg1_dd.value}"
                query(f"desk{d}" + (f" @{at}" if at else ""))
            elif cmd == "trader":
                t = "" if arg1_dd.value == "(all)" else f" {arg1_dd.value}"
                query(f"trader{t}" + (f" @{at}" if at else ""))
            elif cmd == "pnl":
                to_part = "" if arg2_dd.value == "(latest)" else f" {arg2_dd.value}"
                query(f"pnl {arg1_dd.value}{to_part}" + (f" @{at}" if at else ""))
            elif cmd == "pv":
                target = "" if arg1_dd.value == "(all)" else f" {arg1_dd.value}"
                query(f"pv{target}" + (f" @{at}" if at else ""))
            elif cmd == "pvdesk":
                query("pvdesk" + (f" @{at}" if at else ""))
            elif cmd == "summary":
                query("summary" + (f" @{at}" if at else ""))
            elif cmd == "positions":
                query("positions" + (f" @{at}" if at else ""))
        except ValueError as e:
            print(f"Error: {e}")

run_btn.on_click(_run)

# ── Layout ────────────────────────────────────────────────────────────────────
ui = widgets.VBox([
    widgets.HBox([cmd_dd, arg1_dd, arg2_dd]),
    widgets.HBox([event_dd, run_btn]),
    output,
])
display(ui)
